# MATHIVA — Stage-A warm-up on DeepMind Mathematics (100k)

Fine-tunes **flan-t5-base** on the generic DeepMind math set (96k train / 2k val)
to build general math-solving ability, on a **free Colab T4 GPU**. This is the
*warm-up*: afterwards you continue-train the same model on the small General-Math
curriculum set (the other notebook) for the actual `/ask` task.

It is self-contained: upload `deepmind_data.zip` + `train.py`, train, download the model.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Files to have ready on your machine (from `ml/t5/`):
`deepmind/deepmind_data.zip` and `train.py`.

## 1. Install training dependencies

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece

## 2. Upload the data bundle + training script

When the file picker appears, select **`deepmind_data.zip`** and **`train.py`**.
The zip is unpacked into the `data/` layout `train.py` expects.

In [ ]:
import os, zipfile
from google.colab import files

uploaded = files.upload()  # pick deepmind_data.zip, train.py

os.makedirs('data', exist_ok=True)
with zipfile.ZipFile('deepmind_data.zip') as z:
    z.extractall('data')
print('cwd:', os.listdir('.'), '| data/:', os.listdir('data'))

## 3. Confirm the GPU is attached

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — switch runtime to T4 GPU')

## 4. Fine-tune (Stage A)

**3 epochs, not 20** — this set has 96k examples, so a few passes are plenty and
20 would run for hours. Early stopping on val loss still applies. `--batch_size 8`
is safe on a T4; raise to 16 if memory allows. To trade quality for speed, add
`--model_name google/flan-t5-small`.

The best-val checkpoint is saved to `model/`.

In [ ]:
!python train.py --data_dir data --out_dir model --epochs 3 --batch_size 8

## 5. Download the warm-up model (checkpoint stripped)

Bundles only the model files, **not** the training checkpoint/optimizer state, so
the download is ~1 GB instead of ~4 GB. Save this as your Stage-A model — for
Stage B, continue-train it on the curriculum data by passing this folder as
`--model_name` to `train.py`.

In [ ]:
import shutil, os
from google.colab import files

# Bundle ONLY the files needed to run/continue the model. The training run also
# leaves a checkpoint-*/ folder holding a duplicate model + the Adam optimizer
# state (~2-3x the model size) -- useless for inference or Stage B, so we skip it.
# This keeps the download ~1 GB instead of ~4 GB.
os.makedirs('model_final', exist_ok=True)
for f in os.listdir('model'):
    p = os.path.join('model', f)
    if os.path.isfile(p):  # config.json, model.safetensors, tokenizer*, generation_config
        shutil.copy(p, 'model_final')

print('bundling (should be ~5 files, no checkpoint-*):', os.listdir('model_final'))
shutil.make_archive('mathiva_t5_stageA_deepmind', 'zip', 'model_final')
files.download('mathiva_t5_stageA_deepmind.zip')